# ✏️ Hands-On Paper & Pencil Lab: RoPE Multi-Pair Rotation

Welcome! In this notebook, we explore how RoPE handles higher-dimensional vectors.

---

## 1. Why Can't RoPE Use a Dimension of 3?

RoPE rotates in <b>2D planes</b>. A 2D plane requires <b>pairs of 2 coordinates</b> (x, y).

Because of this, in every foundation model (Llama 3, Mirendil-Base, DeepSeek):
<pre style='font-size: 15px; background: #161b22; color: #f85149; padding: 12px; border-radius: 8px;'>
The head dimension d must ALWAYS be an EVEN number (e.g. 64, 128).
An odd number like 3 would leave a number without a pair!
</pre>

A vector of dimension <b>4</b> is split into <b>two 2D pairs</b>:
<pre style='font-size: 15px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
x = [ x₁, x₂ ,   x₃, x₄ ]
     └ Pair 1 ┘   └ Pair 2 ┘
</pre>

---
## 2. Your Pencil & Paper Exercise 📝

Let our 4D vector be:
<pre style='font-size: 16px; background: #161b22; color: #c9d1d9; padding: 12px; border-radius: 8px;'>
x = [ 1.0,  0.0,    0.0,  1.0 ]
     └─ Pair 1 ─┘  └─ Pair 2 ─┘
</pre>

Let rotation angle <b>θ = 90°</b> (cos 90° = 0.0, sin 90° = 1.0).

### Step-by-Step on Paper:

<b>Pair 1: [1.0, 0.0]</b>
- x'₁ = (1.0 · 0.0) - (0.0 · 1.0) = <b>0.0</b>
- x'₂ = (1.0 · 1.0) + (0.0 · 0.0) = <b>1.0</b>

<b>Pair 2: [0.0, 1.0]</b>
- x'₃ = (0.0 · 0.0) - (1.0 · 1.0) = 0.0 - 1.0 = <b>-1.0</b>
- x'₄ = (0.0 · 1.0) + (1.0 · 0.0) = 0.0 + 0.0 = <b>0.0</b>

<br>
<b>Final Rotated 4D Vector:</b><br>
<span style='color: #3fb950; font-size: 16px; font-weight: bold;'>
[ 0.0, 1.0,   -1.0, 0.0 ]
</span>

---
## 3. Verify in PyTorch with Real 4D RoPE Code

Run this cell to see our actual RoPE implementation rotate both pairs simultaneously:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import math
from src.model.rope import apply_rotary_emb

# Input 4D vector: 2 pairs
x = torch.tensor([[[[1.0, 0.0, 0.0, 1.0]]]], dtype=torch.float32)  # [1, 1, 1, 4]

# Angle theta = 90 degrees
theta = math.pi / 2
cos = torch.tensor([[math.cos(theta), math.cos(theta), math.cos(theta), math.cos(theta)]])
sin = torch.tensor([[math.sin(theta), math.sin(theta), math.sin(theta), math.sin(theta)]])

# Rotate using our actual project function
x_rot, _ = apply_rotary_emb(x, x, cos, sin)

print(f"Initial 4D Vector:  {x.squeeze().tolist()}")
print(f"Rotated 4D Vector:  {[round(v, 4) for v in x_rot.squeeze().tolist()]}")